# RetinaScreen AI — Phase 1: Exploratory Data Analysis (EDA)

This notebook performs exploratory analysis on the Diabetic Retinopathy dataset:
1. **Class Distribution**: Verifying class balance across standard 5 ICDR grades.
2. **CLAHE Pairing Check**: Confirming base image vs CLAHE enhancement variants.
3. **Image Quality Heuristic**: Calculating Laplacian variance (blur score) and mean brightness distribution.

In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Mount Drive if running in Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

DATASET_DIR = '/content/drive/MyDrive/DATASETS/Diabetic_Retinopathy_dataset'
CLASSES = ['No_DR', 'Mild', 'Moderate', 'Severe', 'Proliferative_DR']

print('Dataset path:', DATASET_DIR)

In [ ]:
# Class count verification
counts = {}
for split in ['train', 'test']:
    counts[split] = {}
    for cls in CLASSES:
        orig_path = os.path.join(DATASET_DIR, split, 'original', cls)
        clahe_path = os.path.join(DATASET_DIR, split, 'CLAHE', cls)
        num_orig = len(glob.glob(os.path.join(orig_path, '*.*')))
        num_clahe = len(glob.glob(os.path.join(clahe_path, '*.*')))
        counts[split][cls] = {'original': num_orig, 'CLAHE': num_clahe}

df_counts = pd.DataFrame(counts)
print(df_counts)

In [ ]:
# Laplacian Variance (Blur Score) & Brightness Distribution
def analyze_image_quality(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None, None
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    brightness = np.mean(gray)
    return lap_var, brightness

# Sample 100 images for fast quality distribution assessment
sample_files = glob.glob(os.path.join(DATASET_DIR, 'train', 'original', '*', '*.*'))[:100]
qualities = [analyze_image_quality(f) for f in sample_files]
blur_scores = [q[0] for q in qualities if q[0] is not None]
brightness_scores = [q[1] for q in qualities if q[1] is not None]

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.hist(blur_scores, bins=20, color='indigo', alpha=0.7)
plt.title('Laplacian Variance (Blur Score)')
plt.axvline(50, color='red', linestyle='--', label='Quality Gate Threshold (50)')
plt.legend()

plt.subplot(1, 2, 2)
plt.hist(brightness_scores, bins=20, color='teal', alpha=0.7)
plt.title('Mean Pixel Brightness')
plt.axvline(30, color='red', linestyle='--', label='Low Limit (30)')
plt.axvline(225, color='red', linestyle='--', label='High Limit (225)')
plt.legend()
plt.tight_layout()
plt.show()